# 00 - Kiểm tra môi trường chạy CPU

Mục tiêu: In thông tin hệ thống, in phiên bản thư viện, kiểm tra đường dẫn dữ liệu, kiểm tra dung lượng đĩa và ghi báo cáo setup_report.json.

Ghi chú: Môi trường chạy thuần CPU, không dùng bộ xử lý đồ họa rời và không dùng thư viện Deep Learning (chỉ sử dụng Machine Learning cổ điển và đặc trưng ảnh thủ công).

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import platform
from pathlib import Path
import importlib.metadata

report = {
    "system": {},
    "packages": {},
    "imports_ok": {},
    "dataset": {},
    "disk_space": {}
}

print("=== 1. SYSTEM INFORMATION ===")
python_version = sys.version
os_name = f"{os.name} ({platform.system()} {platform.release()})"
cpu_count = os.cpu_count()

ram_info = "không đọc được"
try:
    with open("/proc/meminfo", "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                ram_info = line.split(":", 1)[1].strip()
                break
except Exception:
    ram_info = "không đọc được"

print(f"Python Version: {python_version}")
print(f"OS: {os_name}")
print(f"CPU Count: {cpu_count}")
print(f"RAM: {ram_info}")

report["system"] = {
    "python_version": python_version,
    "os_name": os_name,
    "cpu_count": cpu_count,
    "ram": ram_info
}

In [ ]:
print("=== 2. KIỂM TRA PHIÊN BẢN VÀ IMPORT THƯ VIỆN ===")
package_specs = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scikit-learn", "sklearn"),
    ("scikit-image", "skimage"),
    ("opencv-python-headless", "cv2"),
    ("pillow", "PIL"),
    ("gradio", "gradio"),
]

packages = {}
imports_ok = {}

print("--- Phiên bản gói (importlib.metadata) ---")
for pkg_name, mod_name in package_specs:
    try:
        ver = importlib.metadata.version(pkg_name)
        packages[pkg_name] = ver
        print(f"{pkg_name}=={ver}")
    except Exception:
        packages[pkg_name] = None
        print(f"{pkg_name}: THIẾU")

print("\n--- Kiểm tra import module ---")
for pkg_name, mod_name in package_specs:
    try:
        __import__(mod_name)
        imports_ok[mod_name] = True
        print(f"{mod_name}: import OK")
    except Exception as e:
        imports_ok[mod_name] = False
        print(f"{mod_name}: import LỖI: {e}")

report["packages"] = packages
report["imports_ok"] = imports_ok

In [ ]:
print("=== GỢI Ý requirements.txt (copy nguyên từ chạy thật) ===")
for pkg_name, ver in packages.items():
    if ver is not None:
        print(f"{pkg_name}=={ver}")
    else:
        print(f"# THIẾU: {pkg_name}")

In [ ]:
data_check_skipped = not Path("/kaggle/input").exists()
if data_check_skipped:
    print("=== 3. DATASET INSPECTION UNDER /kaggle/input ===")
    print("Bỏ qua: không phải môi trường Kaggle (không có /kaggle/input)")
else:
    print("=== 3. DATASET INSPECTION UNDER /kaggle/input ===")
    input_root = Path("/kaggle/input")
    
    if not input_root.exists():
        print("WARNING: /kaggle/input does not exist! Running in non-Kaggle environment.")
        input_root = Path(".")
    
    print(f"Inspecting root: {input_root.resolve()}")
    top_level_items = list(input_root.iterdir())
    print("Top-level items found:")
    for item in top_level_items:
        print(f" - {item.name} ({'DIR' if item.is_dir() else 'FILE'})")
    
    # Dynamically locate dataset folder
    dataset_dir = None
    for item in input_root.rglob("*"):
        if item.is_dir() and "covid" in item.name.lower() and "radiography" in item.name.lower():
            dataset_dir = item
            break
    
    if dataset_dir is None and len(top_level_items) > 0:
        # fallback to first top level directory
        for item in top_level_items:
            if item.is_dir():
                dataset_dir = item
                break
    
    print(f"\nDetected Dataset Directory: {dataset_dir}")
    
    dataset_report = {
        "dataset_dir": str(dataset_dir),
        "classes": {},
        "total_images": 0,
        "total_masks": 0
    }
    
    valid_extensions = {'.png', '.jpg', '.jpeg'}
    
    if dataset_dir and dataset_dir.exists():
        # Traverse children of dataset directory
        # Note: structure could be dataset_dir/COVID-19_Radiography_Dataset/{COVID, Normal, ...}
        # or directly dataset_dir/{COVID, Normal, ...}
        inner_dirs = [d for d in dataset_dir.iterdir() if d.is_dir()]
        if len(inner_dirs) == 1 and "radiography" in inner_dirs[0].name.lower():
            target_search_dir = inner_dirs[0]
        else:
            target_search_dir = dataset_dir
        
        print(f"Target search directory: {target_search_dir}")
        for class_folder in sorted(target_search_dir.iterdir()):
            if not class_folder.is_dir():
                continue
            class_name = class_folder.name
            images_count = 0
            masks_count = 0
            subdirs_found = [sub.name for sub in class_folder.iterdir() if sub.is_dir()]
            
            # check for 'images' subdirectory
            images_dir = class_folder / "images"
            if images_dir.exists() and images_dir.is_dir():
                images_count = sum(1 for f in images_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions)
            else:
                # count image files directly in class_folder
                images_count = sum(1 for f in class_folder.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions)
                
            # check for 'masks' subdirectory
            masks_dir = class_folder / "masks"
            if masks_dir.exists() and masks_dir.is_dir():
                masks_count = sum(1 for f in masks_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions)
                
            dataset_report["classes"][class_name] = {
                "folder_name": class_name,
                "subdirectories": subdirs_found,
                "images_count": images_count,
                "masks_count": masks_count
            }
            dataset_report["total_images"] += images_count
            dataset_report["total_masks"] += masks_count
            
            print(f"Class folder: {class_name} | Subdirs: {subdirs_found} | Images: {images_count} | Masks: {masks_count}")
        
        print(f"\nTotal Images Counted: {dataset_report['total_images']}")
        print(f"Total Masks Counted: {dataset_report['total_masks']}")
    else:
        print("Could not locate dataset folder.")
    
    report["dataset"] = dataset_report


In [ ]:
print("=== 5. DISK SPACE INSPECTION ===")
working_dir = Path("/kaggle/working")
if not working_dir.exists():
    print("Bỏ qua: không phải môi trường Kaggle")
else:
    total, used, free = shutil.disk_usage(working_dir)

    total_gb = round(total / (2**30), 2)
    used_gb = round(used / (2**30), 2)
    free_gb = round(free / (2**30), 2)

    print(f"Disk Usage for {working_dir.resolve()}:")
    print(f"  Total: {total_gb} GB")
    print(f"  Used:  {used_gb} GB")
    print(f"  Free:  {free_gb} GB")

    report["disk_space"] = {
        "working_dir": str(working_dir.resolve()),
        "total_gb": total_gb,
        "used_gb": used_gb,
        "free_gb": free_gb
    }

In [ ]:
print("=== 6. WRITE /kaggle/working/setup_report.json ===")
working_dir = Path("/kaggle/working")
if not working_dir.exists():
    print("Bỏ qua: không phải môi trường Kaggle")
else:
    output_report_path = working_dir / "setup_report.json"
    report["packages"] = packages
    report["imports_ok"] = imports_ok

    with open(output_report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"Report saved successfully to {output_report_path.resolve()}.")
    print("\n--- setup_report.json Content Preview ---")
    print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
print("=== 7. TỔNG KẾT ĐÁNH GIÁ ===")
failed_reasons = []

required_modules = ["numpy", "pandas", "sklearn", "skimage", "cv2", "PIL", "gradio"]
for mod in required_modules:
    if not imports_ok.get(mod, False):
        failed_reasons.append(f"Module {mod} import thất bại")

if data_check_skipped:
    print("Kiểm tra dữ liệu: bỏ qua (không phải Kaggle)")
else:
    data_ok = (
        dataset_dir is not None
        and dataset_dir.exists()
        and dataset_report.get("total_images", 0) > 0
    )
    if not data_ok:
        failed_reasons.append("Đường dẫn dữ liệu không hợp lệ hoặc không tìm thấy ảnh")

if not failed_reasons:
    print("KẾT QUẢ: ĐẠT")
else:
    print("KẾT QUẢ: KHÔNG ĐẠT")
    print("Danh sách mục lỗi:")
    for reason in failed_reasons:
        print(f" - {reason}")